In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# 1. Load train.csv using Hugging Face datasets and create combined_text

In [2]:
from datasets import load_dataset

# Load train.csv using Hugging Face datasets
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

# Access training split
train_dataset = dataset["train"]

# Function to combine prompt and option A
def combine_columns(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

# Apply function to every row
train_dataset = train_dataset.map(combine_columns)

# Length of row index 51
length = len(train_dataset[51]["combined_text"])

print(length)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


# 2. Vocabulary size of bert-base-uncased

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


# 3. Token ID of [SEP]

In [4]:
print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


# 4. Tokenize all prompts

In [5]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert prompt column to a list of strings
prompts = [str(text) if text is not None else "" for text in train_dataset["prompt"]]

# Tokenize all prompts
encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Print the shape of input_ids
print(encoded["input_ids"].shape)

torch.Size([2000, 128])


# 5. Dimension of each attention head

In [6]:
hidden_size = 768
attention_heads = 12

head_dimension = hidden_size // attention_heads

print(head_dimension)

64


# 6. Shape of last_hidden_state

In [7]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

text = train_dataset[0]["prompt"]

inputs = tokenizer(
    text,
    return_tensors="pt"
)

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


# 7. Sum of first five values of CLS embedding

In [8]:
cls_embedding = outputs.last_hidden_state[0, 0]

first_five = cls_embedding[:5]

answer = first_five.sum().item()

print(round(answer, 4))

-1.2001


# 8. Attention weight from CLS to "fusion"

In [9]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

sentence = "Light-ion fusion is a technique."

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

outputs = model(**inputs)

# Last layer
last_layer_attention = outputs.attentions[-1]

# First head
first_head = last_layer_attention[0, 0]

# Convert IDs back to tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

fusion_index = tokens.index("fusion")

attention_score = first_head[0, fusion_index].item()

print(round(attention_score, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


# 9. Sentence Transformer similarity

In [10]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt = train_dataset[0]["prompt"]
option_b = train_dataset[0]["B"]

prompt_embedding = embedding_model.encode(
    prompt,
    convert_to_tensor=True
)

option_embedding = embedding_model.encode(
    option_b,
    convert_to_tensor=True
)

similarity = util.cos_sim(
    prompt_embedding,
    option_embedding
)

print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


# 10. TF-IDF Pipeline + MiniLM Pipeline + MAP@3

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from sentence_transformers import util

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

options = ["A", "B", "C", "D", "E"]


# -----------------------------
# MAP@3 function
# -----------------------------
def map_at_3(actual, predicted):

    for rank, prediction in enumerate(predicted[:3], start=1):
        if prediction == actual:
            return 1 / rank

    return 0


tfidf_scores = []
mini_scores = []

improved = 0


for row in train_dataset:

    prompt = row["prompt"]

    option_texts = [
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    correct = row["answer"]


    # -----------------------------
    # TF-IDF
    # -----------------------------
    documents = [prompt] + option_texts

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(documents)

    similarities = cosine_similarity(
        vectors[0:1],
        vectors[1:]
    )[0]

    tfidf_order = similarities.argsort()[::-1]

    tfidf_prediction = [
        options[i]
        for i in tfidf_order
    ]

    tfidf_scores.append(
        map_at_3(correct, tfidf_prediction)
    )


    # -----------------------------
    # MiniLM
    # -----------------------------
    prompt_embedding = model.encode(
        prompt,
        convert_to_tensor=True
    )

    option_embeddings = model.encode(
        option_texts,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]

    mini_order = similarities.argsort(descending=True)

    mini_prediction = [
        options[int(i)]
        for i in mini_order
    ]

    mini_scores.append(
        map_at_3(correct, mini_prediction)
    )


    # Improvement count

    tfidf_top3 = tfidf_prediction[:3]
    mini_top3 = mini_prediction[:3]

    if (
        correct not in tfidf_top3
        and
        correct in mini_top3
    ):
        improved += 1


final_map = sum(mini_scores) / len(mini_scores)

print("MiniLM MAP@3 =", final_map)

print("Improved Count =", improved)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3 = 0.4230833333333333
Improved Count = 564


# 11. Zero-shot classification

In [12]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification"
)

sequence = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"]
]

result = classifier(
    sequence,
    candidate_labels
)

print(result)

print(round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

# 12. Multi-label=True

In [13]:
result_softmax = classifier(
    sequence,
    candidate_labels
)

result_sigmoid = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])

sigmoid_sum = sum(result_sigmoid["scores"])

difference = abs(
    softmax_sum - sigmoid_sum
)

print(round(difference, 4))

0.9995


In [14]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load dataset
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train_dataset = dataset["train"]

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Row 0
row = train_dataset[0]

input_text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(input_text, return_tensors="pt")

# Generate
outputs = model.generate(**inputs, max_new_tokens=5)

# Decode
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B
